# Modelos de recomendação

In [55]:
import pandas as pd
import functions
import spotipy
from time import sleep
import json
import os

from pyspark.sql import SparkSession
os.environ['SPARK_HOME'] = '/home/david/Documentos/UFABC/PGC/Codigos/code/Spark'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'jupyter' 
os.environ['PYSPARK_DRIVER_OPTS'] = 'notebook'
os.environ['PYSPARK_PYTHON'] = 'python'

In [56]:
spark = SparkSession.builder \
        .appName('PGC') \
        .getOrCreate()

ConnectionRefusedError: [Errno 111] Connection refused

## ALS

- [Recommender System using ALS in Pyspark](https://medium.com/@brunoborges_38708/recommender-system-using-als-in-pyspark-10329e1d1ee1)
- [Sistema de recomendação de filmes com filtragem colaborativa usando Spark e ALS](https://medium.com/camilawaltrick/sistema-de-recomendacao-filtragem-colaborativa-als-spark-f5a4a7ccf8cf)
    * [Github do artigo citado a cima](https://github.com/cwaltrick/data_science/blob/master/Modelo_ALS.ipynb)


### Importações necessárias

In [16]:
from pyspark.ml.evaluation import RegressionEvaluator #evaluation é a biblioteca para verificação da qualidade do modelo
from pyspark.ml.recommendation import ALS # ALS é o modelo de recomendação que será utilizadp
from pyspark.sql import Row #row é o formato que o ALS trabalha, row conterá o id do usuario, id filme, nota e timestamp

### Carregando dataset do MovieLens

In [51]:
df = pd.read_csv('datasets/MovieLens/100K/ml-100k/u.data',
                names=['userId', 'itemId', 'rating', 'timestamp'],
                sep='\t')

In [53]:
df = spark.read.txt("datasets/MovieLens/100K/ml-100k/u.data", header = True, inferSchema = True)

ConnectionRefusedError: [Errno 111] Connection refused

### Criando o modelo

#### Separação simples entre treino e teste (80% treino e 20% teste)

In [46]:
(training, test) = df.randomSplit([0.8, 0.2])

#### Modelo ALS

Parâmetros: quantidade máxima de iterações, coeficiente de aprendizado, as colunas utilizadas e desconsidera o usuário que tiver coldstart, caso ocorra.

In [47]:
als = ALS(maxIter=5, regParam=0.01, userCol="userId", itemCol="movieId", ratingCol="rating", coldStartStrategy="drop")

#### Treinando o modelo com a porção de treino

In [48]:
model = als.fit(training)

24/08/05 19:36:20 WARN BlockManager: Block rdd_126_6 could not be removed as it was not found on disk or in memory
24/08/05 19:36:20 WARN BlockManager: Block rdd_127_6 could not be removed as it was not found on disk or in memory
24/08/05 19:36:20 ERROR Executor: Exception in task 6.0 in stage 21.0 (TID 48)
java.lang.OutOfMemoryError: Java heap space
	at java.base/java.util.Arrays.copyOf(Arrays.java:3676)
	at scala.collection.mutable.ArrayBuilder$ofFloat.mkArray(ArrayBuilder.scala:471)
	at scala.collection.mutable.ArrayBuilder$ofFloat.resize(ArrayBuilder.scala:475)
	at scala.collection.mutable.ArrayBuilder$ofFloat.ensureSize(ArrayBuilder.scala:487)
	at scala.collection.mutable.ArrayBuilder$ofFloat.$plus$eq(ArrayBuilder.scala:492)
	at scala.collection.mutable.ArrayBuilder$ofFloat.$plus$eq(ArrayBuilder.scala:462)
	at scala.collection.generic.Growable.$anonfun$$plus$plus$eq$1(Growable.scala:62)
	at scala.collection.generic.Growable$$Lambda/0x00007c285f1ce4d0.apply(Unknown Source)
	at scal

Py4JJavaError: An error occurred while calling o238.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 6 in stage 21.0 failed 1 times, most recent failure: Lost task 6.0 in stage 21.0 (TID 48) (192.168.15.5 executor driver): java.lang.OutOfMemoryError: Java heap space
	at java.base/java.util.Arrays.copyOf(Arrays.java:3676)
	at scala.collection.mutable.ArrayBuilder$ofFloat.mkArray(ArrayBuilder.scala:471)
	at scala.collection.mutable.ArrayBuilder$ofFloat.resize(ArrayBuilder.scala:475)
	at scala.collection.mutable.ArrayBuilder$ofFloat.ensureSize(ArrayBuilder.scala:487)
	at scala.collection.mutable.ArrayBuilder$ofFloat.$plus$eq(ArrayBuilder.scala:492)
	at scala.collection.mutable.ArrayBuilder$ofFloat.$plus$eq(ArrayBuilder.scala:462)
	at scala.collection.generic.Growable.$anonfun$$plus$plus$eq$1(Growable.scala:62)
	at scala.collection.generic.Growable$$Lambda/0x00007c285f1ce4d0.apply(Unknown Source)
	at scala.collection.IndexedSeqOptimized.foreach(IndexedSeqOptim

24/08/05 19:36:24 WARN BlockManager: Putting block rdd_126_8 failed due to exception org.apache.spark.TaskKilledException.
24/08/05 19:36:24 WARN BlockManager: Block rdd_126_8 could not be removed as it was not found on disk or in memory
24/08/05 19:36:24 WARN BlockManager: Putting block rdd_127_8 failed due to exception org.apache.spark.TaskKilledException.
24/08/05 19:36:24 WARN BlockManager: Block rdd_127_8 could not be removed as it was not found on disk or in memory
24/08/05 19:36:24 WARN TaskSetManager: Lost task 8.0 in stage 21.0 (TID 50) (192.168.15.5 executor driver): TaskKilled (Stage cancelled: Job aborted due to stage failure: Task 6 in stage 21.0 failed 1 times, most recent failure: Lost task 6.0 in stage 21.0 (TID 48) (192.168.15.5 executor driver): java.lang.OutOfMemoryError: Java heap space
	at java.base/java.util.Arrays.copyOf(Arrays.java:3676)
	at scala.collection.mutable.ArrayBuilder$ofFloat.mkArray(ArrayBuilder.scala:471)
	at scala.collection.mutable.ArrayBuilder$of

In [ ]:
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating",
predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print("Erro médio quadrático = " + str(rmse))